# Pepper Root Strategy Analysis

This notebook studies whether `INTARIAN_PEPPER_ROOT` can be improved beyond our original Round 1/2 strategy.

Our original strategy already captured the main upward drift by entering long positions early. Here, we first document that baseline, then check whether a small spread-capture or DP-style extension could add value without damaging the existing Root PnL.

## 1) Original Pepper Root Strategy

The original strategy performed well on `INTARIAN_PEPPER_ROOT` because the product moved upward in a very stable way across all three Round 1 days.

The dashboard screenshot below shows that our strategy entered long positions early, which allowed it to capture most of this upward drift.

![Pepper Root price and fills](https://raw.githubusercontent.com/vanshkharbanda93-alt/IMC_Prosperity_4/round1-post-competition-review/round_1/improved_strategies/assets/buy_and_hold_pepperroot.png)

### Pepper Root backtest results

| Matching mode | Pepper Root PnL |
|---|---:|
| `all` | 236,797 |
| `worse` | 236,665 |
| `none` | 236,294 |

The Pepper Root PnL is almost unchanged across matching assumptions of the backtester. This suggests that the Root edge was not mainly coming from optimistic passive-fill assumptions, but from capturing the product's upward drift.

## 2) Round 1 price data

In [ ]:
import numpy as np 
import pandas as pd
import re
import ast

from pathlib import Path

In [3]:
BASE = Path.cwd()

prices_day_minus_2 = pd.read_csv(BASE / "prices_round_1_day_-2.csv", sep=";")
prices_day_minus_1 = pd.read_csv(BASE / "prices_round_1_day_-1.csv", sep=";")
prices_day_0 = pd.read_csv(BASE / "prices_round_1_day_0.csv", sep=";")

In [ ]:
prices = pd.concat(
    [prices_day_minus_2, prices_day_minus_1, prices_day_0],
    ignore_index=True
)

root_data = prices[prices["product"] == "INTARIAN_PEPPER_ROOT"].copy()

In [9]:
root_data["day"].value_counts().sort_index()

day
-2    10000
-1    10000
 0    10000
Name: count, dtype: int64

## 3) Analysis

### 3.1) Spread capture. Yes or no?

In [10]:
slope_rows = []

for day, day_df in root_data.groupby("day"):
    day_df = day_df.sort_values("timestamp")
    
    slope_per_timestamp, intercept = np.polyfit(
        day_df["timestamp"],
        day_df["mid_price"],
        1
    )
    
    slope_per_tick = slope_per_timestamp * 100
    
    slope_rows.append({
        "day": day,
        "start_mid": day_df["mid_price"].iloc[0],
        "end_mid": day_df["mid_price"].iloc[-1],
        "total_mid_change": day_df["mid_price"].iloc[-1] - day_df["mid_price"].iloc[0],
        "slope_per_tick": slope_per_tick,
    })

drift_summary = pd.DataFrame(slope_rows)
drift_summary

,day,start_mid,end_mid,total_mid_change,slope_per_tick
0,-2,9998.5,11001.5,1003.0,0.100105
1,-1,10998.5,11998.0,999.5,0.100855
2,0,11998.5,13000.0,1001.5,0.103285


Since the fair value keeps rising, buying and holding is generally the simplest PnL-generating strategy for Pepper Root. Selling too early can be expensive: even if we sell at a good price now, we may need to buy back later at a higher price.

However, it may still be possible to improve the strategy by selectively capturing the spread. At some points, temporarily reducing our long position may be more beneficial than simply holding, especially if the quoted price is attractive enough.

Question:
- Is the spread wide enough to justify temporarily reducing our long position?

To answer the above question we ask
- If we sell one unit to capture the spread, for how many ticks can we afford to be out of that long position before the upward drift cancels the benefit?

We determine this using the data by computing the (median spread)/(average drift per tick)

In [11]:
root_data = root_data.sort_values(["day", "timestamp"]).copy()

root_data["spread"] = root_data["ask_price_1"] - root_data["bid_price_1"]

average_drift_per_tick = drift_summary["slope_per_tick"].mean()

spread_summary = (
    root_data
    .groupby("day")["spread"]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)

spread_summary["median_spread_in_drift_ticks"] = (
    spread_summary["median"] / average_drift_per_tick
)

spread_summary

,day,mean,median,min,max,median_spread_in_drift_ticks
0,-2,11.994792,12.0,2.0,18.0,118.325624
1,-1,13.012257,13.0,2.0,19.0,128.186093
2,0,14.128715,14.0,2.0,21.0,138.046561


Answer: Spread capture is worth investigating.


The median spread is 12, 13, 14 price units, while the estimated drift is only around 0.1 per tick. This means that one median spread is equivalent to roughly 120, 130, 140 ticks of upward drift.

This suggests that the spread is large enough to make spread capture potentially worthwhile. However, this does not mean we should sell aggressively. The main risk is execution: after selling, we need to buy back quickly enough. Otherwise, the upward drift may erase the benefit of the spread capture.

Since the median spread appears to increase every day, the spread becomes larger relative to the drift cost. In other words, a successful spread-capture has more room to compensate for the upward drift as the days progress. Hence the data suggests that spread capture more worthwhile as the days progress. 

We dont sell whenever possible, we ask:
 
Question: when is the ask price is attractive enough to justify temporarily reducing our long position?

Answer:
The ask price is attractive when the spread we expect to capture is larger than the drift we expect to give up while temporarily reducing our long position.
If we sell at the ask and later buy back at the bid after $h$ ticks, a rough break-even condition is:
$
\text{spread} \approx \mu h
$
where $\mu$ is the estimated drift per tick.

So the sell decision depends on the expected buy-back time. A large spread is valuable only if we can buy back before the upward drift erases the spread advantage.
From the data it is evident that we must capture the buy-back within 120 ticks (if possible much lower) such that the spread capture is larger than the loss by reducing our position.

Question: How often is the spread actually around 12–14 or higher?

In [14]:
spread_thresholds = [8, 10, 12, 14, 16, 18]

spread_frequency_rows = []

for day, day_df in root_data.groupby("day"):
    for threshold in spread_thresholds:
        fraction = (day_df["spread"] >= threshold).mean()
        
        spread_frequency_rows.append({
            "day": day,
            "spread_threshold": threshold,
            "fraction_of_snapshots": fraction,
        })

spread_frequency = pd.DataFrame(spread_frequency_rows)

spread_frequency_pivot = spread_frequency.pivot(
    index="day",
    columns="spread_threshold",
    values="fraction_of_snapshots"
)

spread_frequency_pivot

spread_threshold,8,10,12,14,16,18
day,,,,,,
-2,0.8880,0.8794,0.5734,0.2403,0.0241,0.0025
-1,0.8890,0.8778,0.8747,0.2649,0.1205,0.0196
0,0.8953,0.8882,0.8837,0.5773,0.2732,0.0325


### 3.2) Experimental strategy and buy-back probability from backtest results

The experimental strategy used here is the ``trader_pepperroot_variant.py``

Question: How quickly could we realistically buy back at a good price after reducing the position?
The buy-back is a stochastic event. We cannot know from the quoted spread alone whether a passive bid will be filled. Therefore, if we reduce our long position, what is the probability that we can rebuild it within a given number of ticks?

The quote data tells us whether the spread is large enough to make the trade economically attractive. But execution data or backtest fills are needed to estimate whether the buy-back is likely to happen quickly enough.

However the original strategy only buys early and holds the long position. Therefore, it does not generate sell-and-buy-back events to estimate buy-back probability after temporarily reducing inventory.

To study this properly, we need an experimental trader strategy. The purpose of this strategy is to test whether the strategy can sell a small amount at attractive asks and rebuild the long position quickly enough. The resulting backtest fills can then be used to estimate:

$
P(\text{buy back within } h \text{ ticks} \mid \text{sell fill})
$

for different horizons $h$, such as 25, 50, 100, and 120 ticks.

Experimental Strategy:

Based on the spread/drift analysis, we created a small experimental Root variant to test whether controlled spread capture can add value. The experimental strategy keeps the main long bias because Pepper Root has strong upward drift. However, when the position is already near the limit and the spread is wide, it sells a small quantity. After selling, it tries to rebuild the long position quickly.

The basic behaviour is:

* stay long because the product drifts upward;
* only sell when the spread is sufficiently wide;
* sell only a small quantity;
* after selling, try to buy back toward the long target.


![Pepper Root price and fills](https://raw.githubusercontent.com/vanshkharbanda93-alt/IMC_Prosperity_4/round1-post-competition-review/round_1/improved_strategies/assets/spread_capture_pepperroot.png)

Backtest results

| Matching assumption | Original Root PnL | Experimental Root PnL | Difference |
| ------------- | ----------------: | --------------------: | ---------: |
| `all`         |           236,797 |               238,276 |     +1,479 |
| `worse`       |           236,665 |               238,166 |     +1,501 |
| `none`        |           236,294 |               225,433 |    -10,861 |

The spread-capture variant improves Pepper Root under both `all` and `worse` matching assumptions. This is encouraging because `worse` is stricter than `all`, so the improvement is not only coming from the most optimistic passive-fill assumption.

The variant performs worse under `none`, which is expected. This strategy relies on passive spread-capture fills, while `none` disables passive matching against historical trades. Therefore, `none` is useful as a stress test, but it is not the main evaluation mode for this type of market-making strategy.


Using the backtest we estimate

$
P(\text{buy back within } h \text{ ticks} \mid \text{sell fill})
$

for different horizons $h$, such as 25, 50, 100, and 120 ticks.

In [18]:
log_path = Path("logs/round1_trader_experiment_worse.log")
log_text = log_path.read_text()

In [24]:
for key in ["Sandbox logs", "Activities log", "Trade History"]:
    print(key, log_text.find(key))

print()
idx = log_text.find("Trade History")
print(log_text[idx:idx + 2000])

Sandbox logs 0
Activities log 1968905
Trade History 6691867

Trade History:
[
  {
    "timestamp": 700,
    "buyer": "SUBMISSION",
    "seller": "",
    "symbol": "ASH_COATED_OSMIUM",
    "currency": "XIREC",
    "price": 9992,
    "quantity": 6,
  },
  {
    "timestamp": 900,
    "buyer": "SUBMISSION",
    "seller": "",
    "symbol": "ASH_COATED_OSMIUM",
    "currency": "XIREC",
    "price": 9998,
    "quantity": 8,
  },
  {
    "timestamp": 900,
    "buyer": "",
    "seller": "",
    "symbol": "ASH_COATED_OSMIUM",
    "currency": "XIREC",
    "price": 9998,
    "quantity": 2,
  },
  {
    "timestamp": 1000,
    "buyer": "SUBMISSION",
    "seller": "",
    "symbol": "INTARIAN_PEPPER_ROOT",
    "currency": "XIREC",
    "price": 10004,
    "quantity": 7,
  },
  {
    "timestamp": 2500,
    "buyer": "",
    "seller": "SUBMISSION",
    "symbol": "ASH_COATED_OSMIUM",
    "currency": "XIREC",
    "price": 10010,
    "quantity": 8,
  },
  {
    "timestamp": 4000,
    "buyer": "SUBMISSION",
 

In [32]:
trade_start = log_text.find("Trade History:")

trade_text = log_text[trade_start:]

list_start = trade_text.find("[")
list_end = trade_text.rfind("]") + 1

trade_list_text = trade_text[list_start:list_end]

trades = ast.literal_eval(trade_list_text)

len(trades), trades[:3]

(3444,
 [{'timestamp': 700,
   'buyer': 'SUBMISSION',
   'seller': '',
   'symbol': 'ASH_COATED_OSMIUM',
   'currency': 'XIREC',
   'price': 9992,
   'quantity': 6},
  {'timestamp': 900,
   'buyer': 'SUBMISSION',
   'seller': '',
   'symbol': 'ASH_COATED_OSMIUM',
   'currency': 'XIREC',
   'price': 9998,
   'quantity': 8},
  {'timestamp': 900,
   'buyer': '',
   'seller': '',
   'symbol': 'ASH_COATED_OSMIUM',
   'currency': 'XIREC',
   'price': 9998,
   'quantity': 2}])

In [33]:
trades_df = pd.DataFrame(trades)

trades_df.head()

,timestamp,buyer,seller,symbol,currency,price,quantity
0,700,SUBMISSION,,ASH_COATED_OSMIUM,XIREC,9992,6
1,900,SUBMISSION,,ASH_COATED_OSMIUM,XIREC,9998,8
2,900,,,ASH_COATED_OSMIUM,XIREC,9998,2
3,1000,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10004,7
4,2500,,SUBMISSION,ASH_COATED_OSMIUM,XIREC,10010,8


In [ ]:
trades_df["day_index"] = trades_df["timestamp"] // 1_000_000
trades_df["local_timestamp"] = trades_df["timestamp"] % 1_000_000 

day_map = {
    0: -2,
    1: -1,
    2: 0,
}

trades_df["day"] = trades_df["day_index"].map(day_map)

trades_df[["timestamp", "day", "local_timestamp", "symbol", "buyer", "seller", "price", "quantity"]].head()

,timestamp,day,local_timestamp,symbol,buyer,seller,price,quantity
0,700,-2,700,ASH_COATED_OSMIUM,SUBMISSION,,9992,6
1,900,-2,900,ASH_COATED_OSMIUM,SUBMISSION,,9998,8
2,900,-2,900,ASH_COATED_OSMIUM,,,9998,2
3,1000,-2,1000,INTARIAN_PEPPER_ROOT,SUBMISSION,,10004,7
4,2500,-2,2500,ASH_COATED_OSMIUM,,SUBMISSION,10010,8


In [37]:
trades_df["side"] = None

trades_df.loc[trades_df["buyer"] == "SUBMISSION", "side"] = "BUY"
trades_df.loc[trades_df["seller"] == "SUBMISSION", "side"] = "SELL"

root_fills = trades_df[
    (trades_df["symbol"] == "INTARIAN_PEPPER_ROOT")
    & (trades_df["side"].notna())
].copy()

root_fills = root_fills.sort_values(["day", "local_timestamp"]).reset_index(drop=True)

root_fills.head()

,timestamp,buyer,seller,symbol,currency,price,quantity,day_index,local_timestamp,day,side
0,1000,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10004,7,0,1000,-2,BUY
1,4000,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10010,11,0,4000,-2,BUY
2,4000,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10010,7,0,4000,-2,BUY
3,5200,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10002,5,0,5200,-2,BUY
4,5300,SUBMISSION,,INTARIAN_PEPPER_ROOT,XIREC,10011,11,0,5300,-2,BUY


In [39]:
root_fills.groupby(["day", "side"]).agg(fill_count=("quantity", "count"), total_quantity=("quantity", "sum"))

fill_count  total_quantity
day side                            
-2  BUY           58             233
    SELL          37             153
-1  BUY           79             336
    SELL          57             256
 0  BUY           69             295
    SELL          46             215

Question: How long did it take until the next Root BUY fill happened on the same day?

In [ ]:
sell_fills = root_fills[root_fills["side"] == "SELL"].copy()
buy_fills = root_fills[root_fills["side"] == "BUY"].copy()

buyback_rows = []

for _, sell in sell_fills.iterrows():
    same_day_future_buys = buy_fills[(buy_fills["day"] == sell["day"]) # same day
                                     & (buy_fills["local_timestamp"] > sell["local_timestamp"])] # later time

    if len(same_day_future_buys) == 0:
        ticks_to_next_buy = None
    else:
        next_buy = same_day_future_buys.iloc[0]
        ticks_to_next_buy = (next_buy["local_timestamp"] - sell["local_timestamp"]) / 100

    buyback_rows.append({
        "day": sell["day"],
        "sell_timestamp": sell["local_timestamp"],
        "sell_price": sell["price"],
        "sell_quantity": sell["quantity"],
        "ticks_to_next_buy": ticks_to_next_buy,
    })

buyback_df = pd.DataFrame(buyback_rows)

buyback_df.head()

,day,sell_timestamp,sell_price,sell_quantity,ticks_to_next_buy
0,-2,131400,10138,5,32.0
1,-2,175000,10180,3,99.0
2,-2,231600,10236,4,30.0
3,-2,235900,10243,5,74.0
4,-2,248400,10256,5,16.0


Buy back probability for different horizons $h$, such as 25, 50, 100, and 120 ticks.

$
P(\text{buy back within } h \text{ ticks} \mid \text{sell fill})
$



In [43]:
horizons = [25, 50, 100, 120]

buyback_summary = []

for H in horizons:
    prob = (buyback_df["ticks_to_next_buy"] <= H).mean()
    
    buyback_summary.append({
        "horizon_ticks": H,
        "buyback_probability": prob,
    })

buyback_summary = pd.DataFrame(buyback_summary)
buyback_summary

,horizon_ticks,buyback_probability
0,25,0.350000
1,50,0.557143
2,100,0.814286
3,120,0.850000


The buy-back probability analysis supports the spread-capture idea. Around 85% of Root sell fills were followed by a buy fill within 120 ticks, which is close to the rough break-even window implied by the median spread and drift estimate. This helps explain why the experimental variant improved Root PnL under all/worse matching.

Question: For each day, what fraction of sell fills were bought back within that day’s own break-even horizon?

In [44]:
day_breakeven_horizon = {
    -2: 120,
    -1: 130,
    0: 140,
}

buyback_df["breakeven_horizon"] = buyback_df["day"].map(day_breakeven_horizon)

buyback_df["within_breakeven"] = (
    buyback_df["ticks_to_next_buy"] <= buyback_df["breakeven_horizon"]
)

buyback_df.groupby("day")["within_breakeven"].mean()

day
-2    0.783784
-1    0.929825
 0    0.826087
Name: within_breakeven, dtype: float64

## Conclusion

The spread and drift analysis suggested that small spread-capture trades could add value, as the median spread was large relative to the drift per tick.

The experimental trader tested this idea by selling small quantities only when the spread was wide, while rebuilding the long position afterward. Backtests showed a modest improvement in Pepper Root PnL under both `all` and `worse` matching assumptions.

The fill analysis supports the mechanism: most sell fills were followed by buy-backs within the approximate spread/drift break-even window. This suggests that the improvement was not accidental.

Overall, the original strategy was directionally correct, but a small spread-capture layer could have improved the Root component further.